In [ ]:
import orjson
import logging
from pathlib import Path

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

In [ ]:
# ----------------------------
# Keypoint order (your list)
# ----------------------------
BP = [
    "nose",
    "upper_jaw",
    "lower_jaw",
    "mouth_end_right",
    "mouth_end_left",
    "right_eye",
    "right_earbase",
    "right_earend",
    "right_antler_base",
    "right_antler_end",
    "left_eye",
    "left_earbase",
    "left_earend",
    "left_antler_base",
    "left_antler_end",
    "neck_base",
    "neck_end",
    "throat_base",
    "throat_end",
    "back_base",
    "back_end",
    "back_middle",
    "tail_base",
    "tail_end",
    "front_left_thai",
    "front_left_knee",
    "front_left_paw",
    "front_right_thai",
    "front_right_knee",
    "front_right_paw",
    "back_left_paw",
    "back_left_thai",
    "back_right_thai",
    "back_left_knee",
    "back_right_knee",
    "back_right_paw",
    "belly_bottom",
    "body_middle_right",
    "body_middle_left",
]
IDX = {name: i for i, name in enumerate(BP)}
K = len(BP)
logger.info(f"Keypoint order: {BP}")

In [ ]:
pos_sources = Path("rodent-samples/pos")
assert pos_sources.is_dir(), f"{pos_sources} is not a directory"

pos_samples = list(pos_sources.glob("*.jpg"))
logger.info(f"Found {len(pos_samples)} positive samples in {pos_sources}")

pos_sample = pos_samples[0]
pos_sample_name = pos_sample.stem.replace("positive_", "")
logger.info(f"Example positive sample: {pos_sample_name}")
pos_video_name = pos_sample_name.split("_frame_")[0]
try:
    frame_number = int(pos_sample_name.split("_frame_")[1])
except IndexError, ValueError:
    logger.error(f"Unexpected filename format: {pos_sample_name}")
    frame_number = None

logger.info(f"Corresponding video name: {pos_video_name}")
logger.info(f"Frame number: {frame_number}")

In [ ]:
assert frame_number is not None, "Frame number could not be extracted from filename"

dlc_sources = Path("videos/deeplabcut_inference")
assert dlc_sources.is_dir(), f"{dlc_sources} is not a directory"
dlc_sample = dlc_sources / pos_video_name
dlc_sample_dir = None
experiments = list(dlc_sources.glob("*"))
for experiment in experiments:
    if experiment.is_dir():
        for video_dir in experiment.glob("*"):
            if video_dir.is_dir() and video_dir.name == pos_video_name:
                dlc_sample_dir = video_dir
                break
    if dlc_sample_dir is not None:
        break

if dlc_sample_dir is not None:
    keypoints_json = dlc_sample_dir.glob("*after_adapt.json")
    keypoints_json = list(keypoints_json)[0] if keypoints_json else None
    if keypoints_json is not None:
        try:
            with open(keypoints_json, "rb") as f:
                keypoints_data = orjson.loads(f.read())
            bparts_keypoints = (
                keypoints_data[frame_number]["bodyparts"]
                if frame_number < len(keypoints_data)
                else {}
            )
            for bparts_keypoint in bparts_keypoints:
                for idx, body_part_coords in enumerate(bparts_keypoint):
                    body_part = BP[idx]
                    if not any(
                        [body_part_coord == -1 for body_part_coord in body_part_coords]
                    ):
                        logger.info(f"{body_part}: {body_part_coords}")

        except Exception as e:
            logger.error(f"Error reading keypoints JSON: {e}")